## Ecommerce Data Cleaning
### Step 1 - Load Libraries and Data

In [1]:
import pandas as pd
import numpy as np

orders = pd.read_csv("dataset/olist_orders_dataset.csv")
items = pd.read_csv("dataset/olist_order_items_dataset.csv")
products = pd.read_csv("dataset/olist_products_dataset.csv")
customers = pd.read_csv("dataset/olist_customers_dataset.csv")
payments = pd.read_csv("dataset/olist_order_payments_dataset.csv")
reviews = pd.read_csv("dataset/olist_order_reviews_dataset.csv")
category = pd.read_csv("dataset/product_category_name_translation.csv")

### Step 2 - Inspect Data

In [2]:
print("Orders:", orders.shape)
print("Items:", items.shape)
print("Products:", products.shape)
print("Customers:", customers.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)

Orders: (99441, 8)
Items: (112650, 7)
Products: (32951, 9)
Customers: (99441, 5)
Payments: (103886, 5)
Reviews: (99224, 7)


### Step 3 - Check missing values

In [3]:
for name, df_ in zip(
    ['orders','items','products','customers','payments','reviews'],
    [orders, items, products, customers, payments, reviews]
):
    print(f"\n{name}:\n", df_.isnull().sum())


orders:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

items:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

products:
 product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

customers:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_sta

### Step 4 - Handle missing values

In [4]:
reviews = reviews.drop(columns=['review_comment_title','review_comment_message'])

### Step 5 - Convert date cols

In [5]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

orders['order_estimated_delivery_date'] = pd.to_datetime(
    orders['order_estimated_delivery_date']
)

In [6]:
reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date']
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp']
)

### Step 6 - Check duplicates

In [7]:
print("Orders duplicates:", orders.duplicated().sum())
print("Items duplicates:", items.duplicated().sum())
print("Products duplicates:", products.duplicated().sum())
print("Customers duplicates:", customers.duplicated().sum())
print("Payments duplicates:", payments.duplicated().sum())
print("Reviews duplicates:", reviews.duplicated().sum())

Orders duplicates: 0
Items duplicates: 0
Products duplicates: 0
Customers duplicates: 0
Payments duplicates: 0
Reviews duplicates: 0


In [8]:
payments = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', 'first')
)

### Step 7 - Merge Tables

In [9]:
df = orders.merge(items, on="order_id", how="left")

In [10]:
df = df.merge(products, on="product_id", how="left")

In [11]:
df = df.merge(category, on="product_category_name", how="left")

In [12]:
df = df.merge(customers, on="customer_id", how="left")

In [13]:
df = df.merge(payments, on="order_id", how="left")

In [14]:
final_df = df.merge(reviews, on="order_id", how="left")

### Step 8 - Feature engineering

In [15]:
final_df['revenue'] = final_df['price'] + final_df['freight_value']

In [16]:
final_df['order_month'] = final_df['order_purchase_timestamp'].dt.to_period('M')

In [17]:
final_df['order_year'] = final_df['order_purchase_timestamp'].dt.year

In [18]:
final_df['delivery_days'] = (
    final_df['order_delivered_customer_date'] -
    final_df['order_purchase_timestamp']
).dt.days

### Step 9 - Final cleaning

In [19]:
final_df = final_df[final_df['order_status'] == 'delivered']

In [20]:
final_df['product_category_name_english'] = final_df[
    'product_category_name_english'
].fillna('uncategorized')

In [21]:
final_df.shape

(110840, 37)

In [22]:
final_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,payment_value,payment_type,review_id,review_score,review_creation_date,review_answer_timestamp,revenue,order_month,order_year,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,38.71,credit_card,a54f0611adc9ed256b57ede6b6eb5114,4.0,2017-10-11,2017-10-12 03:43:48,38.71,2017-10,2017,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,141.46,boleto,8d5266042046a06655c8db133d120ba5,4.0,2018-08-08,2018-08-08 18:37:50,141.46,2018-07,2018,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,179.12,credit_card,e73b67b67587f7644d5bd1a52deb1b01,5.0,2018-08-18,2018-08-22 19:07:58,179.12,2018-08,2018,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,72.20,credit_card,359d03e676b3c069f62cadba8dd3f6e8,5.0,2017-12-03,2017-12-05 19:21:58,72.20,2017-11,2017,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,28.62,credit_card,e50934924e227544ba8246aeb3770dd4,5.0,2018-02-17,2018-02-18 13:02:51,28.62,2018-02,2018,2.0


In [23]:
final_df.to_csv('ecom_cleaned.csv',index='False')

### Final checks

In [24]:
final_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
order_estimated_delivery_date       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1545
product_name_lenght              1545
product_description_lenght       1545
product_photos_qty               1545
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_category_name_english       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_sta

In [25]:
final_df['order_status'].value_counts()

order_status
delivered    110840
Name: count, dtype: int64

In [26]:
final_df['revenue'].describe()

count    110840.000000
mean        139.747975
std         188.967970
min           6.080000
25%          55.130000
50%          91.990000
75%         157.340000
max        6929.310000
Name: revenue, dtype: float64

In [27]:
final_df = final_df.dropna(subset=['delivery_days'])

In [28]:
final_df['product_name_lenght'] = final_df['product_name_lenght'].fillna(0)
final_df['product_description_lenght'] = final_df['product_description_lenght'].fillna(0)
final_df['product_photos_qty'] = final_df['product_photos_qty'].fillna(0)

In [29]:
final_df['product_weight_g'] = final_df['product_weight_g'].fillna(
    final_df['product_weight_g'].median()
)

final_df['product_length_cm'] = final_df['product_length_cm'].fillna(
    final_df['product_length_cm'].median()
)

final_df['product_height_cm'] = final_df['product_height_cm'].fillna(
    final_df['product_height_cm'].median()
)

final_df['product_width_cm'] = final_df['product_width_cm'].fillna(
    final_df['product_width_cm'].median()
)

In [30]:
final_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1545
product_name_lenght                 0
product_description_lenght          0
product_photos_qty                  0
product_weight_g                    0
product_length_cm                   0
product_height_cm                   0
product_width_cm                    0
product_category_name_english       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_sta

In [31]:
final_df = final_df.drop(columns=['product_category_name'])

In [32]:
final_df.shape

(110832, 36)

In [33]:
[col for col in final_df.columns if 'review' in col]

['review_id',
 'review_score',
 'review_creation_date',
 'review_answer_timestamp']

In [34]:
final_df['payment_value'] = final_df['payment_value'].fillna(0)
final_df['payment_type'] = final_df['payment_type'].fillna('unknown')

In [38]:
final_df.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 15
order_delivered_carrier_date       1
order_delivered_customer_date      0
order_estimated_delivery_date      0
order_item_id                      0
product_id                         0
seller_id                          0
shipping_limit_date                0
price                              0
freight_value                      0
product_name_lenght                0
product_description_lenght         0
product_photos_qty                 0
product_weight_g                   0
product_length_cm                  0
product_height_cm                  0
product_width_cm                   0
product_category_name_english      0
customer_unique_id                 0
customer_zip_code_prefix           0
customer_city                      0
customer_state                     0
payment_value                      0
p

In [36]:
final_df['review_score'] = final_df['review_score'].fillna(0)
final_df['review_id'] = final_df['review_id'].fillna('no_review')

In [37]:
final_df['review_creation_date'] = final_df['review_creation_date'].fillna(pd.NaT)
final_df['review_answer_timestamp'] = final_df['review_answer_timestamp'].fillna(pd.NaT)

In [39]:
final_df.to_csv("ecomm_cleaned.csv", index=False)